# Evaluating hyperparameter tuning results

In [5]:
import pandas as pd

adult_census = pd.read_csv("datasets/adult-census.csv")
target_name = "class"

target = adult_census[target_name]
data = adult_census.drop(columns=[target_name, "education-num"])

In [6]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import make_column_selector as selector

categorical_columns_selector = selector(dtype_include=object)
categorical_columns = categorical_columns_selector(data)

categorical_preprocessor = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=-1
)
preprocessor = make_column_transformer(
    (categorical_preprocessor, categorical_columns),
    remainder="passthrough",
)

In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline

model = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "classifier",
            HistGradientBoostingClassifier(random_state=42, max_leaf_nodes=4),
        ),
    ]
)

In [8]:
from sklearn.model_selection import cross_validate

cv_results = cross_validate(model, data, target, cv=5)
cv_results = pd.DataFrame(cv_results)
cv_results

,fit_time,score_time,test_score
0,0.189019,0.025899,0.863241
1,0.177574,0.022165,0.860784
2,0.162015,0.022786,0.860360
3,0.158650,0.022995,0.862408
4,0.157184,0.022340,0.866912


In [9]:
from sklearn.model_selection import train_test_split

data_train, data_test, target_train, target_test = train_test_split(
    data, target, test_size=0.2, random_state=42,
)

In [10]:
cv_results = cross_validate(model, data_train, target_train)
cv_results = pd.DataFrame(cv_results)
cv_results

,fit_time,score_time,test_score
0,0.172880,0.027301,0.862188
1,0.176982,0.027332,0.859757
2,0.162167,0.020570,0.864491
3,0.138018,0.019481,0.859867
4,0.145603,0.022757,0.860379


In [11]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__learning_rate": (0.05, 0.5),
    "classifier__max_leaf_nodes": (10, 30),
}

model_grid_search = GridSearchCV(model, param_grid=param_grid, n_jobs=2, cv=2)

In [14]:
cv_results = cross_validate(
    model_grid_search, data, target, cv=5, n_jobs=2, return_estimator=True,
)

In [15]:
for estimator in cv_results["estimator"]:
    print(estimator.best_params_)

{'classifier__learning_rate': 0.05, 'classifier__max_leaf_nodes': 30}
{'classifier__learning_rate': 0.05, 'classifier__max_leaf_nodes': 30}
{'classifier__learning_rate': 0.05, 'classifier__max_leaf_nodes': 30}
{'classifier__learning_rate': 0.05, 'classifier__max_leaf_nodes': 30}
{'classifier__learning_rate': 0.05, 'classifier__max_leaf_nodes': 30}


In [13]:
cv_results = pd.DataFrame(cv_results)
cv_results

,fit_time,score_time,test_score
0,15.982412,0.055427,0.868666
1,16.219131,0.038129,0.869280
2,2.930132,0.040354,0.870188
3,2.854984,0.044261,0.872441
4,1.869240,0.030048,0.874898


In [16]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

data, target = fetch_california_housing(return_X_y=True, as_frame=True)
target *= 100

In [17]:
target

0        452.6
1        358.5
2        352.1
3        341.3
4        342.2
         ...  
20635     78.1
20636     77.1
20637     92.3
20638     84.7
20639     89.4
Name: MedHouseVal, Length: 20640, dtype: float64

In [18]:
data_train, data_test, target_train, target_test = train_test_split(
    data, target, random_state=42,
)

In [19]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

scaler = StandardScaler()

model = make_pipeline(
    scaler,
    KNeighborsRegressor(),
)

In [20]:
data.dtypes

MedInc        float64
HouseAge      float64
AveRooms      float64
AveBedrms     float64
Population    float64
AveOccup      float64
Latitude      float64
Longitude     float64
dtype: object

In [22]:
import numpy as np

from sklearn.model_selection import RandomizedSearchCV

In [27]:
param_distributions = {
    "kneighborsregressor__n_neighbors": np.logspace(0, 3, num=10).astype(np.int32),
    "standardscaler__with_mean": [True, False],
    "standardscaler__with_std": [True, False],
}

In [32]:
model_random_search = RandomizedSearchCV(
    model,
    param_distributions=param_distributions,
    scoring="neg_mean_absolute_error",
    n_iter=20,
    n_jobs=2,
    verbose=0,
    random_state=1,
)

In [33]:
model_random_search.fit(data_train, target_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...Regressor())])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'kneighborsregressor__n_neighbors': array([ 1, ... dtype=int32), 'standardscaler__with_mean': [True, False], 'standardscaler__with_std': [True, False]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratifie

In [31]:
model_random_search.best_params_

{'standardscaler__with_std': True,
 'standardscaler__with_mean': False,
 'kneighborsregressor__n_neighbors': np.int32(10)}

In [35]:
pd.DataFrame(model_random_search.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_standardscaler__with_std,param_standardscaler__with_mean,param_kneighborsregressor__n_neighbors,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.010973,0.000930,0.068833,0.004385,True,False,1,"{'standardscaler__with_std': True, 'standardsc...",-54.775433,-52.588055,-51.359477,-54.828945,-54.631928,-53.636768,1.413792,8
1,0.009719,0.000369,0.066618,0.001286,False,False,215,"{'standardscaler__with_std': False, 'standards...",-91.235919,-88.534179,-91.052123,-90.055153,-90.265650,-90.228605,0.958687,15
2,0.009782,0.000996,0.007853,0.000581,False,False,1,"{'standardscaler__with_std': False, 'standards...",-95.634328,-94.598343,-97.549545,-96.254873,-98.507431,-96.508904,1.382512,20
3,0.009365,0.000391,0.021488,0.003280,False,True,46,"{'standardscaler__with_std': False, 'standards...",-88.821845,-86.048962,-88.889881,-87.346337,-87.225100,-87.666425,1.071938,13
4,0.008770,0.000296,0.031023,0.000213,False,False,100,"{'standardscaler__with_std': False, 'standards...",-90.309987,-87.628587,-90.318812,-89.107234,-89.094267,-89.291777,0.993058,14
5,0.008811,0.000203,0.059843,0.000561,False,True,215,"{'standardscaler__with_std': False, 'standards...",-91.235919,-88.534179,-91.052123,-90.055153,-90.265650,-90.228605,0.958687,15
6,0.009497,0.000774,0.169786,0.014068,True,False,46,"{'standardscaler__with_std': True, 'standardsc...",-48.872857,-45.568601,-45.677369,-46.785742,-47.428248,-46.866563,1.220338,3
7,0.009338,0.000925,0.290640,0.010515,False,False,1000,"{'standardscaler__with_std': False, 'standards...",-92.038302,-89.381648,-91.957675,-91.002378,-91.506284,-91.177257,0.970928,19
8,0.009003,0.000238,0.013560,0.000385,False,False,21,"{'standardscaler__with_std': False, 'standards...",-86.690837,-83.375163,-86.128416,-84.837293,-85.112294,-85.228801,1.144525,11
9,0.009275,0.000256,0.217335,0.008411,True,False,100,"{'standardscaler__with_std': True, 'standardsc...",-50.835264,-47.727923,-48.077773,-49.137860,-49.650534,-49.085871,1.117337,5
